In [41]:
# Libraries.
import pandas as pd
import numpy as np

In [42]:
# Read in data.
gtexEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/gtexExpressionProfile.parquet")
emtabEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/emtabExpressionProfile.parquet")
orthologTable = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/orthologTable.txt", sep="\t")
orthologTableIDs = orthologTable[~orthologTable["Mouse gene stable ID"].isna()].loc[:, ["Gene stable ID", "Mouse gene stable ID"]]

In [43]:
gtexEP

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSG00000000003,5.799691,22.906008,18.107586,3.414238,15.745596,23.313591,7.128223,10.594314,protein_coding
ENSG00000000005,0.169146,0.753748,0.284215,0.258039,0.956031,0.018972,0.050229,0.206583,protein_coding
ENSG00000000419,21.387617,40.514694,43.498722,24.609381,25.236029,22.603436,22.147223,37.476429,protein_coding
ENSG00000000457,2.830432,6.585192,6.015115,1.902403,3.748399,4.093346,3.044870,5.295540,protein_coding
ENSG00000000460,1.347589,2.231597,2.198975,0.689768,0.988887,1.240150,0.699461,1.596039,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSG00000310553,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310554,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310555,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA


In [44]:
emtabEP

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSMUSG00000000001,34.911248,129.180584,143.270691,72.898048,72.920467,45.744554,8.901608,54.003405,protein_coding
ENSMUSG00000000003,0.000000,0.000000,0.000000,0.053683,0.000000,0.000000,0.000000,0.000000,protein_coding
ENSMUSG00000000028,2.236124,11.323525,5.613315,22.680435,2.280480,0.871723,0.181636,3.491172,protein_coding
ENSMUSG00000000031,2.567429,1.137826,2650.257043,17.372786,0.758899,1.307234,1.672216,2.287438,lncRNA
ENSMUSG00000000037,1.226929,2.675817,3.285196,0.774847,0.333726,0.011223,0.000000,0.287549,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSMUSG00000109574,0.332800,0.047076,0.021773,0.015338,0.008382,0.000000,0.000000,0.000000,TEC
ENSMUSG00000109575,1.098742,0.000000,0.000000,0.009073,0.000000,0.000000,0.000000,0.000000,TEC
ENSMUSG00000109576,0.022506,0.000000,0.000000,0.024401,0.012853,0.000000,0.000000,0.000000,TEC


In [45]:
orthologTable

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score
0,ENSG00000198888,ENSMUSG00000064341,ortholog_one2one,50.0
1,ENSG00000198763,ENSMUSG00000064345,ortholog_one2one,75.0
2,ENSG00000198804,ENSMUSG00000064351,ortholog_one2one,100.0
3,ENSG00000198712,ENSMUSG00000064354,ortholog_one2one,100.0
4,ENSG00000228253,ENSMUSG00000064356,ortholog_one2one,100.0
...,...,...,...,...
28009,ENSG00000081692,ENSMUSG00000036819,ortholog_one2one,75.0
28010,ENSG00000157873,ENSMUSG00000022074,ortholog_one2many,0.0
28011,ENSG00000157873,ENSMUSG00000042333,ortholog_one2many,100.0
28012,ENSG00000132676,ENSMUSG00000068921,ortholog_one2one,75.0


In [46]:
orthologTest = tuple(orthologTableIDs.iloc[0, :])
humanOrtholog = gtexEP[gtexEP.index.isin(orthologTest)]
mouseOrtholog = emtabEP[emtabEP.index.isin(orthologTest)]

In [47]:
# Euclidean distance. My own methodology.
def euclideanDist(humanOrtholog, mouseOrtholog):
    distSum = 0
    for i in range(0, humanOrtholog.shape[1] - 1):
        distSum += np.square(humanOrtholog.iloc[0, i] - mouseOrtholog.iloc[0, i])
    return np.sqrt(distSum)

In [48]:
euclideanDist(humanOrtholog, mouseOrtholog)

np.float64(44492.547301776765)

In [49]:
# Euclidean Distance. Method using numpy's euclidean distance formula.
np.linalg.norm(np.array(humanOrtholog.iloc[:, :-1]) - np.array(mouseOrtholog.iloc[:, :-1]))

np.float64(44492.547301776765)

In [50]:
# Pearson Distance. My own methodology.
def pearsonDist(humanOrtholog, mouseOrtholog):
    # Calculates the Z-Scores for our vectors, and uses these Z-Score vectors to calculate Pearson Distance. The formula was provided in Piasecka et al. 2012.
    ZxT = ((humanOrtholog.iloc[:, :-1] - np.mean(humanOrtholog.iloc[:, :-1])) / np.std(humanOrtholog.iloc[:, :-1])).T
    Zy = (mouseOrtholog.iloc[:, :-1] - np.mean(mouseOrtholog.iloc[:, :-1])) / np.std(mouseOrtholog.iloc[:, :-1])
    return 1 - ((Zy.dot(ZxT) / ZxT.shape[0]).iloc[0, 0])

In [51]:
pearsonDist(humanOrtholog, mouseOrtholog)

np.float64(0.2094305670992722)

In [52]:
# Pearson Distance is 1 - r, and I found online that r is just the correlation matrix between two dataframes. So I tried that method.
corr = humanOrtholog.iloc[:, :-1].reset_index(drop=True).corrwith(mouseOrtholog.iloc[:, :-1].reset_index(drop=True), method="pearson", axis=1)
1 - corr[0]

np.float64(0.20943055191829985)

In [53]:
# TEC
def TEC(humanOrtholog, mouseOrtholog):
    # Turns the vectors binary. So if the expression is greater than 1, we consider that "expressed."
    humanOrthoBinary = (humanOrtholog.iloc[:, :-1] > 1).iloc[0, :]
    mouseOrthoBinary = (mouseOrtholog.iloc[:, :-1] > 1).iloc[0, :]

    humanOnlyTissueNum = ((humanOrthoBinary ^ mouseOrthoBinary) & humanOrthoBinary).sum()
    mouseOnlyTissueNum = ((mouseOrthoBinary ^ humanOrthoBinary) & mouseOrthoBinary).sum()

    return ((humanOnlyTissueNum / 8) + (mouseOnlyTissueNum / 8)) / 2


In [54]:
# This cell just gets the distance and TEC values, so I can append them to the expression profiles later.
myEuclideanDistArr = []
myPearsonDistArr = []
myTECArr = []
for i in range(0, orthologTableIDs.shape[0]):
    orthologTest = tuple(orthologTableIDs.iloc[i, :])
    humanOrtholog = gtexEP[gtexEP.index.isin(orthologTest)]
    mouseOrtholog = emtabEP[emtabEP.index.isin(orthologTest)]

    if not humanOrtholog.empty and not mouseOrtholog.empty:
        myEuclideanDistArr.append((orthologTest[0], orthologTest[1], euclideanDist(humanOrtholog, mouseOrtholog)))
        myPearsonDistArr.append((orthologTest[0], orthologTest[1], pearsonDist(humanOrtholog, mouseOrtholog)))
        myTECArr.append((orthologTest[0], orthologTest[1], TEC(humanOrtholog, mouseOrtholog)))

In [55]:
myEuclidDistDF = pd.DataFrame(myEuclideanDistArr, columns=["Human ID", "Mouse ID", "EuclidDist"])
myPearDistDF = pd.DataFrame(myPearsonDistArr, columns=["Human ID", "Mouse ID", "PearDist"])
myTECDF = pd.DataFrame(myTECArr, columns=["Human ID", "Mouse ID", "TEC"])

In [56]:
# The next 4 cells merge the expression profile dataframe with each distance and TEC column.
orthologTableEuclid = orthologTable.merge(myEuclidDistDF.loc[:, ["Human ID", "EuclidDist"]].groupby("Human ID").mean().reset_index(), left_on="Gene stable ID", right_on="Human ID", how="outer").set_index("Human ID")
orthologTableEuclidPear = orthologTableEuclid.merge(myPearDistDF.loc[:, ["Human ID", "PearDist"]].groupby("Human ID").mean().reset_index(), left_on="Gene stable ID", right_on="Human ID", how="outer").set_index("Human ID")
orthologTableEuclidPearTEC = orthologTableEuclidPear.merge(myTECDF.loc[:, ["Human ID", "TEC"]].groupby("Human ID").mean().reset_index(), left_on="Gene stable ID", right_on="Human ID", how="outer").set_index("Human ID")
orthologTableEuclidPearTEC.to_csv("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.csv", index=False)
orthologTableEuclidPearTEC.to_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.parquet", index=False)

In [57]:
orthologTableEuclidPearTEC

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,PearDist,TEC
Human ID,,,,,,,
ENSG00000000003,ENSG00000000003,ENSMUSG00000067377,ortholog_one2one,100.0,44.054842,1.130272,0.0000
ENSG00000000005,ENSG00000000005,ENSMUSG00000031250,ortholog_one2one,100.0,5.172072,1.141118,0.1250
ENSG00000000419,ENSG00000000419,ENSMUSG00000078919,ortholog_one2one,100.0,188.287732,1.211158,0.0000
ENSG00000000457,ENSG00000000457,ENSMUSG00000026584,ortholog_one2one,100.0,37.114733,0.814870,0.0000
ENSG00000000460,ENSG00000000460,ENSMUSG00000041406,ortholog_one2one,0.0,9.113600,0.959842,0.1875
...,...,...,...,...,...,...,...
NaN,ENSG00000310576,ENSMUSG00000035595,ortholog_one2one,100.0,NaN,NaN,NaN
NaN,ENSG00000310579,NaN,NaN,NaN,NaN,NaN,NaN
NaN,ENSG00000310583,NaN,NaN,NaN,NaN,NaN,NaN


In [58]:
gtexEP

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSG00000000003,5.799691,22.906008,18.107586,3.414238,15.745596,23.313591,7.128223,10.594314,protein_coding
ENSG00000000005,0.169146,0.753748,0.284215,0.258039,0.956031,0.018972,0.050229,0.206583,protein_coding
ENSG00000000419,21.387617,40.514694,43.498722,24.609381,25.236029,22.603436,22.147223,37.476429,protein_coding
ENSG00000000457,2.830432,6.585192,6.015115,1.902403,3.748399,4.093346,3.044870,5.295540,protein_coding
ENSG00000000460,1.347589,2.231597,2.198975,0.689768,0.988887,1.240150,0.699461,1.596039,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSG00000310553,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310554,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310555,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA


In [59]:
gtexEP

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSG00000000003,5.799691,22.906008,18.107586,3.414238,15.745596,23.313591,7.128223,10.594314,protein_coding
ENSG00000000005,0.169146,0.753748,0.284215,0.258039,0.956031,0.018972,0.050229,0.206583,protein_coding
ENSG00000000419,21.387617,40.514694,43.498722,24.609381,25.236029,22.603436,22.147223,37.476429,protein_coding
ENSG00000000457,2.830432,6.585192,6.015115,1.902403,3.748399,4.093346,3.044870,5.295540,protein_coding
ENSG00000000460,1.347589,2.231597,2.198975,0.689768,0.988887,1.240150,0.699461,1.596039,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSG00000310553,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310554,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310555,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA


In [60]:
gtexEP.iloc[:, :-1] / np.linalg.norm(gtexEP.iloc[:, :-1], axis=0)

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach
Gene stable ID,,,,,,,,
ENSG00000000003,3.388606e-05,0.000203,0.000190,0.000019,0.000088,1.639152e-04,3.918809e-05,0.000080
ENSG00000000005,9.882726e-07,0.000007,0.000003,0.000001,0.000005,1.333923e-07,2.761384e-07,0.000002
ENSG00000000419,1.249622e-04,0.000359,0.000455,0.000137,0.000141,1.589222e-04,1.217565e-04,0.000283
ENSG00000000457,1.653746e-05,0.000058,0.000063,0.000011,0.000021,2.877984e-05,1.673946e-05,0.000040
ENSG00000000460,7.873608e-06,0.000020,0.000023,0.000004,0.000006,8.719356e-06,3.845351e-06,0.000012
...,...,...,...,...,...,...,...,...
ENSG00000310553,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000
ENSG00000310554,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000
ENSG00000310555,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000


In [61]:
gtexEPNormalized = gtexEP.iloc[:, :-1] / np.linalg.norm(gtexEP.iloc[:, :-1], axis=0)
gtexEPNormalized["Gene type"] = gtexEP["Gene type"]
gtexEPNormalized

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSG00000000003,3.388606e-05,0.000203,0.000190,0.000019,0.000088,1.639152e-04,3.918809e-05,0.000080,protein_coding
ENSG00000000005,9.882726e-07,0.000007,0.000003,0.000001,0.000005,1.333923e-07,2.761384e-07,0.000002,protein_coding
ENSG00000000419,1.249622e-04,0.000359,0.000455,0.000137,0.000141,1.589222e-04,1.217565e-04,0.000283,protein_coding
ENSG00000000457,1.653746e-05,0.000058,0.000063,0.000011,0.000021,2.877984e-05,1.673946e-05,0.000040,protein_coding
ENSG00000000460,7.873608e-06,0.000020,0.000023,0.000004,0.000006,8.719356e-06,3.845351e-06,0.000012,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSG00000310553,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,lncRNA
ENSG00000310554,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,lncRNA
ENSG00000310555,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,lncRNA


In [62]:
emtabEPNormalized = emtabEP.iloc[:, :-1] / np.linalg.norm(emtabEP.iloc[:, :-1], axis=0)
emtabEPNormalized["Gene type"] = emtabEP["Gene type"]
emtabEPNormalized

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSMUSG00000000001,1.010093e-03,0.003495,2.647237e-03,9.011842e-04,7.456336e-04,5.572821e-04,3.727208e-05,0.000265,protein_coding
ENSMUSG00000000003,0.000000e+00,0.000000,0.000000e+00,6.636402e-07,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,protein_coding
ENSMUSG00000000028,6.469815e-05,0.000306,1.037182e-04,2.803813e-04,2.331859e-05,1.061975e-05,7.605325e-07,0.000017,protein_coding
ENSMUSG00000000031,7.428386e-05,0.000031,4.896924e-02,2.147668e-04,7.759972e-06,1.592535e-05,7.001764e-06,0.000011,lncRNA
ENSMUSG00000000037,3.549896e-05,0.000072,6.070112e-05,9.578854e-06,3.412447e-06,1.367259e-07,0.000000e+00,0.000001,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSMUSG00000109574,9.628957e-06,0.000001,4.023099e-07,1.896115e-07,8.571282e-08,0.000000e+00,0.000000e+00,0.000000,TEC
ENSMUSG00000109575,3.179009e-05,0.000000,0.000000e+00,1.121653e-07,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,TEC
ENSMUSG00000109576,6.511668e-07,0.000000,0.000000e+00,3.016546e-07,1.314263e-07,0.000000e+00,0.000000e+00,0.000000,TEC


In [74]:
myEuclideanDistNormArr = []
myPearsonDistNormArr = []
for i in range(0, orthologTableIDs.shape[0]):
    orthologTest = tuple(orthologTableIDs.iloc[i, :])
    humanOrtholog = gtexEPNormalized[gtexEPNormalized.index.isin(orthologTest)]
    mouseOrtholog = emtabEPNormalized[emtabEPNormalized.index.isin(orthologTest)]

    if not humanOrtholog.empty and not mouseOrtholog.empty:
        myEuclideanDistNormArr.append((orthologTest[0], orthologTest[1], euclideanDist(humanOrtholog, mouseOrtholog)))
        myPearsonDistNormArr.append((orthologTest[0], orthologTest[1], pearsonDist(humanOrtholog, mouseOrtholog)))

myEuclidDistNormDF = pd.DataFrame(myEuclideanDistNormArr, columns=["Human ID", "Mouse ID", "EuclidDistNorm"])
myPearDistNormDF = pd.DataFrame(myPearsonDistNormArr, columns=["Human ID", "Mouse ID", "PearDistNorm"])


orthologTableEuclidPearTECNorm = orthologTableEuclidPearTEC.merge(myEuclidDistNormDF.loc[:, ["Human ID", "EuclidDistNorm"]].groupby("Human ID").mean().reset_index(), left_on="Gene stable ID", right_on="Human ID", how="outer").set_index("Human ID")

In [75]:
orthologTableEuclidPearTECNorm

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,PearDist,TEC,EuclidDistNorm
Human ID,,,,,,,,
ENSG00000000003,ENSG00000000003,ENSMUSG00000067377,ortholog_one2one,100.0,44.054842,1.130272,0.0000,0.001212
ENSG00000000005,ENSG00000000005,ENSMUSG00000031250,ortholog_one2one,100.0,5.172072,1.141118,0.1250,0.000097
ENSG00000000419,ENSG00000000419,ENSMUSG00000078919,ortholog_one2one,100.0,188.287732,1.211158,0.0000,0.003195
ENSG00000000457,ENSG00000000457,ENSMUSG00000026584,ortholog_one2one,100.0,37.114733,0.814870,0.0000,0.001000
ENSG00000000460,ENSG00000000460,ENSMUSG00000041406,ortholog_one2one,0.0,9.113600,0.959842,0.1875,0.000181
...,...,...,...,...,...,...,...,...
NaN,ENSG00000310576,ENSMUSG00000035595,ortholog_one2one,100.0,NaN,NaN,NaN,NaN
NaN,ENSG00000310579,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NaN,ENSG00000310583,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
newColOrder = list(orthologTableEuclidPearTECNorm.columns[:5]) + list(orthologTableEuclidPearTECNorm.columns[-1:]) + list(orthologTableEuclidPearTECNorm.columns[5:7])
orthologTableEuclidPearTECNorm = orthologTableEuclidPearTECNorm.loc[:, newColOrder]

In [81]:
orthologTableEuclidPearTECNorm.to_csv("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.csv", index=False)
orthologTableEuclidPearTECNorm.to_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.parquet", index=False)

In [82]:
orthologTableEuclidPearTECNorm

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,EuclidDistNorm,PearDist,TEC
Human ID,,,,,,,,
ENSG00000000003,ENSG00000000003,ENSMUSG00000067377,ortholog_one2one,100.0,44.054842,0.001212,1.130272,0.0000
ENSG00000000005,ENSG00000000005,ENSMUSG00000031250,ortholog_one2one,100.0,5.172072,0.000097,1.141118,0.1250
ENSG00000000419,ENSG00000000419,ENSMUSG00000078919,ortholog_one2one,100.0,188.287732,0.003195,1.211158,0.0000
ENSG00000000457,ENSG00000000457,ENSMUSG00000026584,ortholog_one2one,100.0,37.114733,0.001000,0.814870,0.0000
ENSG00000000460,ENSG00000000460,ENSMUSG00000041406,ortholog_one2one,0.0,9.113600,0.000181,0.959842,0.1875
...,...,...,...,...,...,...,...,...
NaN,ENSG00000310576,ENSMUSG00000035595,ortholog_one2one,100.0,NaN,NaN,NaN,NaN
NaN,ENSG00000310579,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NaN,ENSG00000310583,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [86]:
gtexEPLog = np.log2(gtexEP.iloc[:, :-1] + 1)
emtabEPLog = np.log(emtabEP.iloc[:, :-1] + 1)

In [97]:
gtexEP

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSG00000000003,5.799691,22.906008,18.107586,3.414238,15.745596,23.313591,7.128223,10.594314,protein_coding
ENSG00000000005,0.169146,0.753748,0.284215,0.258039,0.956031,0.018972,0.050229,0.206583,protein_coding
ENSG00000000419,21.387617,40.514694,43.498722,24.609381,25.236029,22.603436,22.147223,37.476429,protein_coding
ENSG00000000457,2.830432,6.585192,6.015115,1.902403,3.748399,4.093346,3.044870,5.295540,protein_coding
ENSG00000000460,1.347589,2.231597,2.198975,0.689768,0.988887,1.240150,0.699461,1.596039,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSG00000310553,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310554,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310555,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA


In [85]:
gtexEPLog

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach
Gene stable ID,,,,,,,,
ENSG00000000003,2.765469,4.579301,4.256073,2.142164,4.065710,4.603691,3.022940,3.535346
ENSG00000000005,0.225455,0.810441,0.360887,0.331176,0.967929,0.027115,0.070704,0.270927
ENSG00000000419,4.484629,5.375550,5.475692,4.678600,4.713478,4.560925,4.532767,5.265903
ENSG00000000457,1.937507,2.923186,2.810467,1.537248,2.247441,2.348614,2.016093,2.654330
ENSG00000000460,1.231180,1.692248,1.677610,0.756825,0.991961,1.163596,0.765077,1.376312
...,...,...,...,...,...,...,...,...
ENSG00000310553,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSG00000310554,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSG00000310555,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [96]:
emtabEPLog

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach
Gene stable ID,,,,,,,,
ENSMUSG00000000001,3.581051,4.868923,4.971691,4.302686,4.302990,3.844698,2.292697,4.007395
ENSMUSG00000000003,0.000000,0.000000,0.000000,0.052291,0.000000,0.000000,0.000000,0.000000
ENSMUSG00000000028,1.174376,2.511510,1.889085,3.164649,1.187990,0.626859,0.166900,1.502114
ENSMUSG00000000031,1.271845,0.759789,7.882789,2.910871,0.564688,0.836049,0.982908,1.190109
ENSMUSG00000000037,0.800624,1.301775,1.455166,0.573714,0.287976,0.011161,0.000000,0.252741
...,...,...,...,...,...,...,...,...
ENSMUSG00000109574,0.287282,0.046001,0.021540,0.015222,0.008347,0.000000,0.000000,0.000000
ENSMUSG00000109575,0.741338,0.000000,0.000000,0.009032,0.000000,0.000000,0.000000,0.000000
ENSMUSG00000109576,0.022256,0.000000,0.000000,0.024108,0.012771,0.000000,0.000000,0.000000


In [91]:
myEuclideanDistLogArr = []
for i in range(0, orthologTableIDs.shape[0]):
    orthologTest = tuple(orthologTableIDs.iloc[i, :])
    humanOrtholog = gtexEPLog[gtexEPLog.index.isin(orthologTest)]
    mouseOrtholog = emtabEPLog[emtabEPLog.index.isin(orthologTest)]

    if not humanOrtholog.empty and not mouseOrtholog.empty:
        myEuclideanDistLogArr.append((orthologTest[0], orthologTest[1], euclideanDist(humanOrtholog, mouseOrtholog)))

myEuclidDistLogDF = pd.DataFrame(myEuclideanDistLogArr, columns=["Human ID", "Mouse ID", "EuclidDistLog"])


orthologTableEuclidPearTECNormLog = orthologTableEuclidPearTECNorm.merge(myEuclidDistLogDF.loc[:, ["Human ID", "EuclidDistLog"]].groupby("Human ID").mean().reset_index(), left_on="Gene stable ID", right_on="Human ID", how="outer").set_index("Human ID")

In [93]:
orthologTableEuclidPearTECNormLog.insert(6, "EuclidDistLog", orthologTableEuclidPearTECNormLog.pop("EuclidDistLog"))

In [95]:
orthologTableEuclidPearTECNormLog.to_csv("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.csv", index=False)
orthologTableEuclidPearTECNormLog.to_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.parquet", index=False)